# Concise, mathematical and few-shot structured CoT (F1–F3) on the computing tasks

`experiments/2026-11-25-concise-math-cot/README.md`. Qwen3.5-2B, thinking off, 16 frames `uniform128`, the gold
sheet as csv, greedy, up to 2,048 new tokens, final answer scored: the structured-CoT pilot's settings, with three
reasoning prompts (`mvp.benchmarks.cot_variants`):

| Condition | `--reasoning` | Adds to the one before |
|---|---|---|
| F1 | `cot_f1` | (vs the pilot's D) brief instructions, no compulsory check against the video, fixed output lines |
| F2 | `cot_f2` | the geometric operation and formula of the task |
| F3 | `cot_f3` | one synthetic worked example of the task |

Only the computing tasks (absolute distance, relative distance, object size). Questions: `STAGE` questions per task
from explore200's 75 computing questions (the development set), nested: a larger stage reuses a smaller one's answers.
A, C, D and E are the pilot's predictions, not run again.

| `STAGE` | Questions per task | Questions | Inferences, F1–F3 |
|---|---:|---:|---:|
| `smoke` (default) | 1 | 3 | 9 |
| `small` | 5 | 15 | 45 |
| `extended` | 10 | 30 | 90 |
| `full` | 25 | 75 | 225 |

Above `small` the notebook stops unless `CONFIRM_LARGER = True`. **The smoke stage** takes about 15–25 minutes on an
L4: installing and the model's download 8–12, the gold sheets 1, the videos 1–2, nine replies 3–10 (a reply that runs
to 2,048 tokens takes about 2 minutes). The timing of each call is saved, so the smoke stage gives the estimate for
the next one.

**After a disconnect** run it again from the top: answers are saved to Drive batch by batch, and each condition's
folder continues only under the same configuration (prompt fingerprint and version, model, settings, sheets, seed). A
changed prompt needs a new `RUN_NAME`.

**Files on Google Drive**, in `MyDrive/mvp/`: `mvp-3d-vlm-cot-followup.zip` (the code; its comment is the commit).
VSI-Bench's questions, its labels and the videos are downloaded here. Results: `MyDrive/mvp/cot_followup/<RUN_NAME>/`.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/mvp"
CODE_ZIP = f"{DRIVE_DIR}/mvp-3d-vlm-cot-followup.zip"
RUN_NAME = "dev_v1"                      # a new name for a new prompt version or configuration
RESULTS_DIR = f"{DRIVE_DIR}/cot_followup/{RUN_NAME}"
MODEL = "Qwen/Qwen3.5-2B"
VARIANTS = ["cot_f1", "cot_f2", "cot_f3"]  # any of them
STAGE = "smoke"                          # smoke | small | extended | full (questions per task: 1, 5, 10, 25)
CONFIRM_LARGER = False                   # needed above "small"
SEED = 0                                 # the subset's order; another seed is another configuration
PROMPT_VERSION = "2026-11-25.1"          # must be cot_variants.PROMPT_VERSION in the code zip
POOL = "dev"                             # dev: explore200's computing questions; or a path to an ids file
VSI_REPO, VSI_REVISION = "nyu-visionx/VSI-Bench", "3a6a1cf9ac54c403509b15260897cc6ddc41b408"
VSI_SHA256 = "aa3ff8cc2010e555f4f865ff96483812ec0b275005b68562f53bb114ad4db9db"
INDEX_SHA256 = "22e0ee1ed3400a5ca59bc49c3791f433265115b4301c183b9772e4e014f5573c"  # build_gold_sheets.py, all 288 scenes
OBJECTS_SHA256 = "bc0214c926be913606031474df123b2d6ee749fba39f799d56f4e8b1b21f41a5"

import hashlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
CODE = Path("/content/mvp-3d-vlm")
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(CODE)
    CODE_COMMIT = z.comment.decode() or "unknown"
X = CODE / "experiments/2026-11-25-concise-math-cot"
assert (X / "followup.py").is_file(), f"{CODE_ZIP} has no experiments/2026-11-25-concise-math-cot/"
reqs = [l.split("#")[0].strip() for l in (CODE / "requirements.txt").read_text().splitlines()]
reqs = [r for r in reqs if r and not r.startswith("--extra-index-url") and r.split("==")[0] not in ("torch", "torchvision")]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[r for r in reqs if r != "decord"]], check=True)
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decord"]).returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "eva-decord"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(CODE)], check=True)
sys.path[:0] = [str(CODE / "src"), str(X)]
ENV = {**os.environ, "PYTHONPATH": str(CODE / "src")}
import torch
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > L4 or A100"
GPU = torch.cuda.get_device_name(0)
from mvp.benchmarks import cot_variants
assert cot_variants.PROMPT_VERSION == PROMPT_VERSION, f"the code's prompts are version {cot_variants.PROMPT_VERSION}"
import followup
K = followup.STAGES[STAGE]
assert K <= followup.STAGES["small"] or CONFIRM_LARGER, f"STAGE {STAGE!r} is {3 * K * len(VARIANTS)} inferences: set CONFIRM_LARGER = True"
print(f"code {CODE_COMMIT}; GPU {GPU}; {STAGE}: {K} per task, {3 * K} questions, {3 * K * len(VARIANTS)} inferences")

## Questions, gold sheets and videos

`test.jsonl` at the pinned revision; the gold sheets of all 288 scenes rebuilt (`build_gold_sheets.py`, about a minute;
their hashes must be the gold-sheet experiment's, which also makes the runs' configuration the same in every session);
the subset; the videos of its scenes; and a check that the eval shows each question the frames the pilot's runs saw.

In [ ]:
import numpy as np
from huggingface_hub import HfFileSystem, hf_hub_download

RAW, VGGT = CODE / "data/raw/VSI-Bench", CODE / "data/vggt"
RAW.mkdir(parents=True, exist_ok=True)
shutil.copy(hf_hub_download(VSI_REPO, "test.jsonl", repo_type="dataset", revision=VSI_REVISION), RAW / "test.jsonl")
assert hashlib.sha256((RAW / "test.jsonl").read_bytes()).hexdigest() == VSI_SHA256, "not the expected test.jsonl"
SHEETS = CODE / "tasks/openvocab_adapter/out_full"
r = subprocess.run([sys.executable, str(CODE / "experiments/2026-11-19-gold-sheet-relations/build_gold_sheets.py"),
                    "--questions", str(RAW / "test.jsonl"), "--prompts",
                    str(CODE / "experiments/2026-11-19-gold-sheet-relations/prompts_full.jsonl"), "--out", str(SHEETS)],
                   cwd=CODE, env=ENV, capture_output=True, text=True)
print(r.stdout[-400:])
assert r.returncode == 0 and f"index.csv sha256 {INDEX_SHA256}" in r.stdout and f"objects.csv sha256 {OBJECTS_SHA256}" in r.stdout, \
    "the gold sheets are not the expected ones: " + r.stderr[-2000:]

QUESTIONS = followup.load_questions(RAW / "test.jsonl")
pool = followup.dev_pool(QUESTIONS) if POOL == "dev" else [int(x) for x in Path(POOL).read_text().split()]
IDS = followup.nested_subset(pool, QUESTIONS, K, SEED)
SCENES = sorted({f"{QUESTIONS[q]['dataset']}/{QUESTIONS[q]['scene_name']}" for q in IDS})
fs = HfFileSystem()
for dataset in sorted({s.split("/")[0] for s in SCENES}):
    todo = [s for s in SCENES if s.startswith(dataset + "/") and not (RAW / f"{s}.mp4").exists()]
    if todo:
        with fs.open(f"datasets/{VSI_REPO}@{VSI_REVISION}/{dataset}.zip", "rb", block_size=2**20) as f, zipfile.ZipFile(f) as z:
            for s in todo:
                (RAW / s).parent.mkdir(parents=True, exist_ok=True)
                (RAW / f"{s}.mp4").write_bytes(z.read(f"{s}.mp4"))
from decord import VideoReader, cpu
for s in SCENES:  # frame_indices.npy as the VGGT export takes them: min(128, N) frames by np.linspace
    n = len(VideoReader(str(RAW / f"{s}.mp4"), ctx=cpu(0)))
    (VGGT / s).mkdir(parents=True, exist_ok=True)
    np.save(VGGT / s / "frame_indices.npy", np.linspace(0, n - 1, num=min(128, n), dtype=int).astype(np.int32))
from mvp.eval.vsibench import frame_indices
pilot = {r["id"]: r for r in map(json.loads, (CODE / "results/zeroshot/vsibench/structured_cot/explore200/gold_cot.jsonl").read_text().splitlines())}
video = lambda q: f"{RAW}/{QUESTIONS[q]['dataset']}/{QUESTIONS[q]['scene_name']}.mp4"
bad = [q for q in IDS if q in pilot and frame_indices(video(q), "uniform128", 16, VGGT) != pilot[q]["frame_indices"]]
assert not bad, f"the eval would show other frames than the pilot's runs for {bad}"
print(f"{len(IDS)} questions {IDS}, {len(SCENES)} videos; the frames are the pilot's")

## Runs

`followup.run` per condition: it checks the folder's configuration (a changed prompt, model, setting, sheet or seed
stops it), logs the exact prompts (`prompts.jsonl`), answers only the questions not answered yet, and times the call
(`timing.jsonl`).

In [ ]:
import time
for variant in VARIANTS:
    t0 = time.time()
    run_dir, cmd, todo = followup.run(variant, K, SHEETS, RESULTS_DIR, pool, SEED, RAW / "test.jsonl", MODEL,
                                      vggt_root=VGGT, prompt_version=PROMPT_VERSION, env=ENV)
    print(f"{variant}: {len(todo)} new questions answered in {(time.time() - t0) / 60:.1f} min -> {run_dir}")

## Prompts and replies, for reading

The text the model got after the video (the video's frames and timestamps come before it), its reply, the answer the
final-answer rule read, and its score.

In [ ]:
from mvp.benchmarks.vsibench import final_letter, final_number, MCA_QUESTION_TYPES
SHOW_PROMPTS = True
for variant in VARIANTS:
    d = Path(RESULTS_DIR) / variant
    prompts = {r["id"]: r for r in map(json.loads, (d / "prompts.jsonl").read_text().splitlines())}
    preds = {r["id"]: r for r in map(json.loads, (d / "predictions.jsonl").read_text().splitlines())}
    for q in IDS:
        p = preds.get(q)
        print("=" * 100)
        print(f"{variant} | question {q} ({QUESTIONS[q]['question_type']}) | truth {QUESTIONS[q]['ground_truth']}")
        if SHOW_PROMPTS:
            print("--- prompt after the video ---\n" + prompts[q]["after_video"])
        if p is None:
            print("--- (no reply yet) ---")
            continue
        read = final_letter(p["prediction"]) if p["question_type"] in MCA_QUESTION_TYPES else final_number(p["prediction"])
        print(f"--- reply ({p.get('n_new_tokens')} tokens{', CUT OFF' if p.get('hit_token_limit') else ''}) ---\n"
              f"{p['prediction']}\n--- read: {read}; score {p['score']:.2f}")
for variant in VARIANTS:
    t = Path(RESULTS_DIR) / variant / "timing.jsonl"
    if t.is_file():
        print(variant, [json.loads(l) for l in t.read_text().splitlines()])

## Analysis (exploratory below 25 questions per task)

F1–F3 against the pilot's A, C, D and E on the same questions (`analysis.py`): computing score and per task, parse
rate, reply length (median, 90th percentile), cut-off rate, seconds per question, paired bootstrap intervals, sign
tests with Holm within each family, and the automatic error flags (`errors.csv`, with columns to fill in by reading).

In [ ]:
import analysis
text, runs, comps, errors = analysis.report(RESULTS_DIR, SHEETS)
print(text)
print(f"saved {RESULTS_DIR}/analysis.md and errors.csv")